# Polars-BigQuery Microbenchmark Comparison

This notebook is a microbenchmark comparing the way of interacting with BigQuery previously recommended at https://docs.pola.rs/user-guide/io/bigquery/ with the same methods in the polars-bigquery Python package.

In [1]:
PROJECT_ID = 'swast-scratch'

In [2]:
import platform
import psutil

# 1. Thread and Core Count
# logical=True gives the number of threads; logical=False gives physical cores
threads = psutil.cpu_count(logical=True)
physical_cores = psutil.cpu_count(logical=False)

# 2. CPU Speed / Frequency
cpu_freq = psutil.cpu_freq()

print("--- Basic Machine Info ---")
print(f"Operating System: {platform.system()}")
print(f"OS Release:       {platform.release()}")
print(f"OS Version:       {platform.version()}")
print(f"Architecture:     {platform.architecture()[0]}")
print()
print("--- CPU Thread & Speed Info ---")
print(f"Processor:        {platform.processor()}")
print(f"Physical Cores:   {physical_cores}")
print(f"Total Threads:    {threads}")
print(f"Current Speed:    {cpu_freq.current:.2f} MHz")


--- Basic Machine Info ---
Operating System: Linux
OS Release:       7.1.6-1rodete1-amd64
OS Version:       #1 SMP PREEMPT_DYNAMIC Debian 7.1.6-1rodete1 (2026-08-05)
Architecture:     64bit

--- CPU Thread & Speed Info ---
Processor:        
Physical Cores:   32
Total Threads:    64
Current Speed:    2449.99 MHz


In [3]:
import polars as pl
from google.cloud import bigquery

bq = bigquery.Client(project='swast-scratch')

In [4]:
import polars_bigquery

plbq = polars_bigquery.Client(quota_project_id='swast-scratch')

## Read (SQL query)

In [5]:
import polars as pl
from google.cloud import bigquery

bq = bigquery.Client(project='swast-scratch')

query = """
SELECT name, number, year
FROM bigquery-public-data.usa_names.usa_1910_2013
WHERE STARTS_WITH(name, 'T')
AND number > 10
AND year = 2000 
"""

In [6]:
%%timeit
pyarrow_table = bq.query(query).to_arrow()
bq_df = pl.from_arrow(pyarrow_table)

1.76 s ± 205 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [7]:
import polars_bigquery

plbq = polars_bigquery.Client(quota_project_id='swast-scratch')

query = """
SELECT name, number, year
FROM bigquery-public-data.usa_names.usa_1910_2013
WHERE STARTS_WITH(name, 'T')
AND number > 10
AND year = 2000 
"""

In [8]:
%%timeit
plbq_df = plbq.read_query(query)

The slowest run took 4.40 times longer than the fastest. This could mean that an intermediate result is being cached.
768 ms ± 428 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Scan (table)

In [10]:
%%timeit
bq_df = (
    pl.from_arrow(bq.list_rows('bigquery-public-data.usa_names.usa_1910_2013').to_arrow())
    .select(pl.col("name"), pl.col("number"), pl.col("year"))
    .filter(
        pl.col("name").str.starts_with("T")
        & (pl.col("number") > 10)
        & (pl.col("year") == 2000)
    )
)


2.62 s ± 200 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [11]:
%%timeit
plbq_df = (
    plbq.scan_table('bigquery-public-data.usa_names.usa_1910_2013')
    .select(pl.col("name"), pl.col("number"), pl.col("year"))
    .filter(
        pl.col("name").str.starts_with("T")
        & (pl.col("number") > 10)
        & (pl.col("year") == 2000)
    ).collect()
)


876 ms ± 74.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
